In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pyttsx3
import threading
import time

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [2]:
last_spoken = {"text": "", "time": 0}
COOLDOWN = 2.5  # seconds before repeating same phrase

def speak(text):
    now = time.time()
    if text == last_spoken["text"] and (now - last_spoken["time"]) < COOLDOWN:
        return
    last_spoken["text"] = text
    last_spoken["time"] = now

    def _run():
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1.0)
        engine.say(text)
        engine.runAndWait()

    threading.Thread(target=_run, daemon=True).start()

# Test it
speak("Text to speech is working!")
time.sleep(2)
print("✅ TTS ready!")

✅ TTS ready!


In [3]:
FINGER_TIPS = [4, 8, 12, 16, 20]
FINGER_PIPS = [3, 7, 11, 15, 19]

def get_fingers(landmarks):
    fingers = []

    # Thumb (uses x-axis)
    fingers.append(landmarks[4].x < landmarks[3].x)

    # Other 4 fingers (uses y-axis — lower y = higher on screen)
    for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:]):
        fingers.append(landmarks[tip].y < landmarks[pip].y)

    return fingers  # [thumb, index, middle, ring, pinky]

def classify_gesture(landmarks):
    f = get_fingers(landmarks)

    # f = [thumb, index, middle, ring, pinky]
    if f == [True,  True,  True,  True,  True ]: return "Open Hand",    "Hello!"
    if f == [False, False, False, False, False ]: return "Fist",         "Stop!"
    if f == [True,  False, False, False, False ]: return "Thumbs Up",    "Great job!"
    if f == [False, True,  False, False, False ]: return "Pointing Up",  "Number One!"
    if f == [False, True,  True,  False, False ]: return "Peace",        "Peace!"
    if f == [False, True,  False, False, True  ]: return "Rock On",      "Rock On!"
    if f == [True,  False, False, False, True  ]: return "Call Me",      "Call Me!"
    if f == [False, True,  True,  True,  True  ]: return "Four Fingers", "Four!"
    if f == [True,  True,  True,  True,  False ]: return "Four Fingers", "Four!"

    return "Unknown", None

print("✅ Gesture classifier ready!")

✅ Gesture classifier ready!


In [4]:
mp_hands    = mp.solutions.hands
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

COLORS = {
    "Open Hand":    (0,   220, 120),
    "Fist":         (0,   60,  220),
    "Thumbs Up":    (0,   200, 255),
    "Pointing Up":  (255, 200, 0  ),
    "Peace":        (200, 0,   255),
    "Rock On":      (0,   140, 255),
    "Call Me":      (255, 100, 0  ),
    "Four Fingers": (100, 255, 100),
    "Unknown":      (80,  80,  80 ),
    "No Hand":      (60,  60,  60 ),
}

def draw_hud(frame, gesture, speech):
    h, w = frame.shape[:2]
    color = COLORS.get(gesture, (150, 150, 150))

    # Dark top bar
    bar = frame.copy()
    cv2.rectangle(bar, (0, 0), (w, 90), (10, 10, 10), -1)
    cv2.addWeighted(bar, 0.55, frame, 0.45, 0, frame)

    # Gesture label
    cv2.putText(frame, f"Gesture: {gesture}", (15, 45),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, color, 2, cv2.LINE_AA)

    # Speech label
    if speech:
        cv2.putText(frame, f'Says: "{speech}"', (15, 78),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (210, 210, 210), 1, cv2.LINE_AA)

    # Quit hint
    cv2.putText(frame, "Press Q to quit", (15, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1, cv2.LINE_AA)

    # Colored border
    cv2.rectangle(frame, (0, 0), (w - 1, h - 1), color, 3)


# ── Open webcam ──
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.65
) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame   = cv2.flip(frame, 1)          # Mirror view
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = hands.process(rgb)
        rgb.flags.writeable = True

        gesture = "No Hand"
        speech  = None

        if results.multi_hand_landmarks:
            for hand_lm in results.multi_hand_landmarks:
                # Draw skeleton
                mp_drawing.draw_landmarks(
                    frame, hand_lm,
                    mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style()
                )
                gesture, speech = classify_gesture(hand_lm.landmark)
                if speech:
                    speak(speech)

        draw_hud(frame, gesture, speech)
        cv2.imshow("Hand Gesture Recognition — Press Q to quit", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("✅ Session ended.")

I0000 00:00:1775316034.258929       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


✅ Session ended.


In [ ]:
# Run this to find the finger pattern of any gesture you make.
# Then add it to classify_gesture() in Cell 3.

cap = cv2.VideoCapture(0)
print("Hold your gesture steady... detecting in 3 seconds.")
time.sleep(3)

with mp_hands.Hands(min_detection_confidence=0.75) as hands:
    for _ in range(60):
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if results.multi_hand_landmarks:
            f = get_fingers(results.multi_hand_landmarks[0].landmark)
            labels = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
            print("\n Finger states:")
            for name, state in zip(labels, f):
                print(f"  {name}: {'UP ✅' if state else 'DOWN ❌'}")
            print(f"\n Add to classify_gesture():")
            print(f"  if f == {f}: return \"MyGesture\", \"My phrase!\"")
            break

cap.release()
cv2.destroyAllWindows()

Hold your gesture steady... detecting in 3 seconds.


I0000 00:00:1775316364.024350       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4



 Finger states:
  Thumb: DOWN ❌
  Index: UP ✅
  Middle: UP ✅
  Ring: UP ✅
  Pinky: UP ✅

 Add to classify_gesture():
  if f == [False, True, True, True, True]: return "MyGesture", "My phrase!"


: 